# Hansen Ch.10 习题完整解答（计算）

**Chapter 10 Resampling Methods**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch10_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：
- 10.1 jackknife 与公式核对
- **10.28** Nerlove
- **10.29** MRW
- **10.30** CPS 小样本 $\theta$
- **10.31** DDK cluster bootstrap + BCa

> **写给只学过李子奈/陈强的同学：** 本章是"用计算机模拟抽样分布"——第 7–9 章**解析地**推渐近分布，本章**用重抽样**算 SE/CI/$p$ 值，不必推公式。两类方法：**jackknife**（删一算散布）、**bootstrap**（重抽模拟分布）。
> **四条原则：** (1) jackknife vs bootstrap 的区别；(2) CI 四层次 percentile→BC→BCa→percentile-$t$（精度递增，默认 BCa）；(3) **bootstrap 检验必须在 $H_0$ 下重抽**（否则临界值被信号撑大、过度保守）；(4) bootstrap 不能修识别错误（$E[Xe]\ne0$ 时覆盖 plim 而非真参数）。
> **核心警示（已 MC 验证）：** 检验 $H_0:\beta=0$，当 $T=9.06$（极显著）时，无约束 bootstrap 临界值≈11.91 ⇒ 荒谬地"不拒绝"；施加 $H_0$ 的临界值≈1.52 ⇒ 正确拒绝。


## 公共函数

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def ols(y, X):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    return beta, e

def hc3_var(X, e):
    n, k = X.shape
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    u = X * (e / np.clip(1.0 - h, 1e-12, None))[:, None]
    return (n / (n - k)) * XXinv @ (u.T @ u) @ XXinv

def jackknife_se_theta(estimator_fn, n):
    """estimator_fn(i) -> theta without observation i"""
    thetas = np.array([estimator_fn(i) for i in range(n)])
    return float(np.sqrt((n - 1) / n * np.sum((thetas - thetas.mean()) ** 2))), thetas

def pairs_bootstrap_theta(X, y, theta_fn, B=999, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    out = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        out[b] = theta_fn(X[idx], y[idx])
    return out

def bc_interval(theta_hat, theta_star, alpha=0.05):
    p0 = np.mean(theta_star <= theta_hat)
    z0 = stats.norm.ppf(np.clip(p0, 1e-8, 1 - 1e-8))
    za = stats.norm.ppf(alpha / 2)
    zb = stats.norm.ppf(1 - alpha / 2)
    a1 = stats.norm.cdf(2 * z0 + za)
    a2 = stats.norm.cdf(2 * z0 + zb)
    return np.quantile(theta_star, [a1, a2]), p0, z0

def bca_interval(theta_hat, theta_star, theta_jack, alpha=0.05):
    # Efron acceleration from jackknife
    th = np.asarray(theta_jack)
    tbar = th.mean()
    num = np.sum((tbar - th) ** 3)
    den = 6.0 * (np.sum((tbar - th) ** 2) ** 1.5) + 1e-18
    a = num / den
    p0 = np.mean(theta_star <= theta_hat)
    z0 = stats.norm.ppf(np.clip(p0, 1e-8, 1 - 1e-8))
    def x(alp):
        z = stats.norm.ppf(alp)
        return stats.norm.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))
    a1, a2 = x(alpha / 2), x(1 - alpha / 2)
    a1, a2 = np.clip([a1, a2], 1e-6, 1 - 1e-6)
    return np.quantile(theta_star, [a1, a2]), a, z0, p0

print("helpers ready")


## Exercise 10.1 核对：jackknife 方差 = 常规 $s^2/n$

In [ ]:

rng = np.random.default_rng(0)
Y = rng.normal(size=50)
r = 3
mu = np.mean(Y ** r)
# formula
V_jack = np.sum((Y ** r - mu) ** 2) / (len(Y) * (len(Y) - 1))
# direct leave-one-out
n = len(Y)
ths = np.array([(np.sum(Y**r) - Y[i]**r) / (n - 1) for i in range(n)])
V_direct = (n - 1) / n * np.sum((ths - ths.mean()) ** 2)
print(V_jack, V_direct, abs(V_jack - V_direct))


## Exercise 10.28 Nerlove：系数与 $\theta=\beta_3+\beta_4+\beta_5$

In [ ]:

ner = pd.read_excel(ROOT / "Nerlove1963/Nerlove1963.xlsx")
for c in ner.columns:
    ner[c] = pd.to_numeric(ner[c], errors="coerce")
ner = ner.dropna().reset_index(drop=True)
y = np.log(ner.Cost.values)
X = np.column_stack([
    np.ones(len(ner)),
    np.log(ner.output.values),
    np.log(ner.Plabor.values),
    np.log(ner.Pcapital.values),
    np.log(ner.Pfuel.values),
])
names = ["const", "logQ", "logPL", "logPK", "logPF"]
n, k = X.shape
beta, e = ols(y, X)
V = hc3_var(X, e)
se_as = np.sqrt(np.diag(V))
print(pd.DataFrame({"beta": beta, "SE_asym(HC3)": se_as}, index=names))

# jackknife SE for beta
def beta_without(i):
    return ols(np.delete(y, i), np.delete(X, i, axis=0))[0]

Bjack = np.array([beta_without(i) for i in range(n)])
se_jack = np.sqrt((n - 1) / n * np.sum((Bjack - Bjack.mean(0)) ** 2, axis=0))
print("jackknife SE:", se_jack)

# bootstrap SE for beta
B = 999
rng = np.random.default_rng(1)
Bboot = np.zeros((B, k))
for b in range(B):
    idx = rng.integers(0, n, n)
    Bboot[b] = ols(y[idx], X[idx])[0]
se_boot = Bboot.std(axis=0, ddof=1)
print("bootstrap SE:", se_boot)

# theta
R = np.array([0.0, 0.0, 1.0, 1.0, 1.0])
theta = R @ beta
se_theta_as = float(np.sqrt(R @ V @ R))
th_jack = Bjack @ R
se_theta_jack = float(np.sqrt((n - 1) / n * np.sum((th_jack - th_jack.mean()) ** 2)))
th_boot = Bboot @ R
se_theta_boot = float(th_boot.std(ddof=1))
print(f"\ntheta={theta:.4f}")
print(f"SE asym={se_theta_as:.4f}, jack={se_theta_jack:.4f}, boot={se_theta_boot:.4f}")
pc = np.quantile(th_boot, [0.025, 0.975])
bca, a_hat, z0, p0 = bca_interval(theta, th_boot, th_jack)
print("percentile 95%:", pc)
print(f"BCa 95%: {bca}, a={a_hat:.4f}, z0={z0:.4f}, p*={p0:.3f}")


## Exercise 10.29 MRW： unrestricted + $\theta=$ 第2–4 系数和

In [ ]:

mrw = pd.read_excel(ROOT / "MRW1992/MRW1992.xlsx")
m = mrw[mrw.N == 1].reset_index(drop=True)
y = (np.log(m.Y85) - np.log(m.Y60)).to_numpy()
X = np.column_stack([
    np.log(m.Y60),
    np.log(m.invest / 100),
    np.log(m.pop_growth / 100 + 0.05),
    np.log(m.school / 100),
    np.ones(len(m)),
])
names = ["logY60", "logI", "log(n+g+d)", "logSchool", "const"]
n, k = X.shape
beta, e = ols(y, X)
V = hc3_var(X, e)
print(pd.DataFrame({"beta": beta, "SE_asym": np.sqrt(np.diag(V))}, index=names))

Bjack = np.array([ols(np.delete(y, i), np.delete(X, i, axis=0))[0] for i in range(n)])
se_jack = np.sqrt((n - 1) / n * np.sum((Bjack - Bjack.mean(0)) ** 2, axis=0))
print("jack SE", se_jack)

B = 999
rng = np.random.default_rng(2)
Bboot = np.zeros((B, k))
for b in range(B):
    idx = rng.integers(0, n, n)
    Bboot[b] = ols(y[idx], X[idx])[0]
print("boot SE", Bboot.std(0, ddof=1))

R = np.array([0.0, 1.0, 1.0, 1.0, 0.0])
theta = R @ beta
th_jack = Bjack @ R
th_boot = Bboot @ R
print(f"theta={theta:.4f}")
print("SE asym", float(np.sqrt(R @ V @ R)),
      "jack", float(np.sqrt((n-1)/n*np.sum((th_jack-th_jack.mean())**2))),
      "boot", float(th_boot.std(ddof=1)))
print("percentile", np.quantile(th_boot, [0.025, 0.975]))
bc, p0, z0 = bc_interval(theta, th_boot)
print(f"BC 95%: {bc}, p*={p0:.3f}, z0={z0:.3f}")


## Exercise 10.30 CPS：从未结婚 + Midwest 白人男性西班牙裔，$n=99$

In [ ]:

df = pd.read_excel(ROOT / "cps09mar/cps09mar.xlsx")
df["experience"] = df.age - df.education - 6
df["lwage"] = np.log(df.earnings / (df.hours * df.week))
df["exp2"] = (df.experience ** 2) / 100
s = df[
    (df.race == 1) & (df.female == 0) & (df.hisp == 1)
    & (df.marital == 7) & (df.region == 2)
].copy().reset_index(drop=True)
print("n =", len(s))
y = s.lwage.to_numpy(float)
X = np.column_stack([s.education, s.experience, s.exp2, np.ones(len(s))])
# theta = b1 / (b2 + 0.2*b3)

def theta_from_beta(b):
    return b[0] / (b[1] + 0.2 * b[2])

beta, e = ols(y, X)
V = hc3_var(X, e)
th = theta_from_beta(beta)
# delta
den = beta[1] + 0.2 * beta[2]
g = np.array([1 / den, -beta[0] / den**2, -0.2 * beta[0] / den**2, 0.0])
se_as = float(np.sqrt(g @ V @ g))
print("beta", beta, "theta", th, "SE_asym", se_as)

n = len(y)
th_jack = []
for i in range(n):
    bi = ols(np.delete(y, i), np.delete(X, i, axis=0))[0]
    th_jack.append(theta_from_beta(bi))
th_jack = np.array(th_jack)
se_jack = float(np.sqrt((n - 1) / n * np.sum((th_jack - th_jack.mean()) ** 2)))
print("SE_jack", se_jack)

B = 1999
rng = np.random.default_rng(3)
th_boot = np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)
    th_boot[b] = theta_from_beta(ols(y[idx], X[idx])[0])
se_boot = float(th_boot.std(ddof=1))
print("SE_boot", se_boot)
print("Note: large gaps among SEs are common with n=99 and a ratio functional.")
bc, p0, z0 = bc_interval(th, th_boot)
print(f"BC 95% CI: {bc}, p*={p0:.3f}")


## Exercise 10.31 DDK：cluster bootstrap SE + BCa

In [ ]:

ddk = pd.read_excel(ROOT / "DDK2011/DDK2011.xlsx")
for c in ddk.columns:
    ddk[c] = pd.to_numeric(ddk[c], errors="coerce")
ddk["ystd"] = (ddk.totalscore - ddk.totalscore.mean()) / ddk.totalscore.std()
d = ddk[["ystd", "tracking", "agetest", "girl", "etpteacher", "percentile", "schoolid"]].dropna().reset_index(drop=True)
y = d.ystd.to_numpy()
X = np.column_stack([d.tracking, d.agetest, d.girl, d.etpteacher, d.percentile, np.ones(len(d))])
names = ["tracking", "age", "girl", "etpteacher", "percentile", "intercept"]
beta, e = ols(y, X)
print("point estimates", dict(zip(names, beta.round(4))))

# cluster map
from collections import defaultdict
rows = defaultdict(list)
for i, g in enumerate(d.schoolid.to_numpy()):
    rows[g].append(i)
schools = np.array(list(rows.keys()))
G = len(schools)

B = 999
rng = np.random.default_rng(4)
Bboot = np.zeros((B, X.shape[1]))
for b in range(B):
    draw = rng.choice(schools, size=G, replace=True)
    idx = np.concatenate([rows[g] for g in draw])
    Bboot[b] = ols(y[idx], X[idx])[0]
se_boot = Bboot.std(axis=0, ddof=1)
print(pd.DataFrame({"beta": beta, "cluster_boot_SE": se_boot}, index=names))

# BCa per coefficient using delete-cluster jackknife for a
def beta_del_cluster(g_del):
    idx = np.concatenate([rows[g] for g in schools if g != g_del])
    return ols(y[idx], X[idx])[0]

# jackknife delete-cluster (may be slow but G~111)
Jack = np.array([beta_del_cluster(g) for g in schools])
for j, nm in enumerate(names):
    bca, a, z0, p0 = bca_interval(beta[j], Bboot[:, j], Jack[:, j])
    print(f"{nm:12s} BCa95% [{bca[0]:.4f}, {bca[1]:.4f}]  a={a:.4f} p*={p0:.3f}")


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch10：jackknife 对样本均值 $=s/\sqrt n$；以及**核心警示**——bootstrap 假设检验必须在 $H_0$ 下重抽（否则临界值被信号撑大、过度保守）。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(10)

# Ex 10.1: jackknife 对样本均值的 SE = 常规 s/√n
n = 50
Y = rng.standard_normal(n) * 2 + 3
loos = np.array([(Y.sum() - Y[i]) / (n - 1) for i in range(n)])
se_jack = np.sqrt((n - 1) / n * ((loos - loos.mean())**2).sum())
print(f"[10.1] jackknife SE={se_jack:.6f} = s/√n={Y.std(ddof=1)/np.sqrt(n):.6f}")

# Ex 10.14 / 10.22: bootstrap 检验必须施加 H₀
# 单侧 H₀:β=0 vs H₁:>0。当 β̂ 较大时，无约束 bootstrap 的 T*=β̂*/se* 分布集中在 β̂/se，
# 其 0.95 分位被信号撑大 ⇒ 巨大的 T 也"不拒绝"（荒谬）。施加 H₀(β=0) 则临界值≈1.645。
print("\n[10.14/10.22] bootstrap 检验：无约束 vs 施加 H₀")
for beta_true in [0.0, 0.3, 1.0]:
    n, B = 100, 999
    X = rng.standard_normal(n)
    Y = beta_true * X + rng.standard_normal(n)
    bhat = np.sum(X*Y) / np.sum(X**2)
    eh = Y - X * bhat
    T = bhat / np.sqrt(np.sum(eh**2) / (n - 1) / np.sum(X**2))
    Tb = np.empty(B); Tb0 = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n); Xb = X[idx]
        # 无约束: 重抽 (Y,X), T* = β̂*/se*（未中心化）
        Yb = Y[idx]
        bb = np.sum(Xb*Yb) / np.sum(Xb**2)
        s2b = np.sum((Yb - Xb*bb)**2) / (n - 1)
        Tb[b] = bb / np.sqrt(s2b / np.sum(Xb**2))
        # 施加 H₀: DGP 中 β=0，即 Y⁰ = 重抽的残差
        Y0 = eh[idx]
        bb0 = np.sum(Xb*Y0) / np.sum(Xb**2)
        s2b0 = np.sum((Y0 - Xb*bb0)**2) / (n - 1)
        Tb0[b] = bb0 / np.sqrt(s2b0 / np.sum(Xb**2))
    cv_u = np.quantile(Tb, 0.95)
    cv_0 = np.quantile(Tb0, 0.95)
    print(f"  β={beta_true}: T={T:5.2f} | 无约束 cv={cv_u:5.2f} (拒绝? {T > cv_u})"
          f" | H₀施加 cv={cv_0:5.2f} (拒绝? {T > cv_0})")
print("  ⇒ 无约束 cv 随 T 增大 → 巨大 T 也'不拒绝'(荒谬); H₀施加 cv≈1.645 ⇒ 正确拒绝。")